# How to page past 10,000 results in a Lucene-style search API

Why search APIs stop at 10,000 results per query and the date-window loop that gets every record anyway.

Read it with charts at https://www.quantisnow.com/sec-api/guides/paginate-past-10000-results. More notebooks: https://github.com/quantisnow/sec-edgar-notebooks. Set `QUANTISNOW_API_KEY` before running; make a key at https://www.quantisnow.com/sec-api/keys.

Every search endpoint returns at most 50 records a page, and one query can page through at most 10,000 of them. Past that, `total.relation` reads `gte` and `from` stops working. Most search APIs that sit on a Lucene index have the same rule, and the way around it is to split the query into windows that each hold fewer than 10,000 records.

## Split by date

In [ ]:
import os, time, requests
from datetime import date, timedelta

URL = "https://www.quantisnow.com/api/v1/filings"
KEY = os.environ["QUANTISNOW_API_KEY"]

def windows(start, end, days):
    d = start
    while d <= end:
        yield d, min(d + timedelta(days=days - 1), end)
        d += timedelta(days=days)

def fetch(query):
    out, start = [], 0
    while True:
        res = requests.post(URL, headers={"Authorization": f"Bearer {KEY}"},
                            json={"query": query, "from": start, "size": 50})
        if res.status_code == 429 and res.json()["error"]["code"] == "rate_limited":
            time.sleep(int(res.headers.get("Retry-After", "1")))
            continue
        res.raise_for_status()
        page = res.json()
        if page["total"]["relation"] == "gte":
            raise ValueError("window too wide: " + query)
        out += page["data"]
        start += 50
        if start >= page["total"]["value"]:
            return out

all_8ks = []
for a, b in windows(date(2025, 1, 1), date(2025, 12, 31), 7):
    all_8ks += fetch(f'formType:"8-K" AND filedAt:[{a} TO {b}]')
print(len(all_8ks))

A week of 8-Ks is about 1,100 filings, well under the limit. Form 4s run closer to 4,000 a week, so use a day or two for those. Raising instead of quietly truncating when a window hits `gte` is what keeps the loop honest.

## Need everything?

If you want a whole dataset rather than a slice, ask about a data license with bulk files at quantisnow@quantisnow.com. They are cheaper for both of us than ten thousand page requests.